# RoboReviews — Exploratory Data Analysis

This notebook is the first deliverable: it explores the Datafiniti consumer-reviews dataset and motivates the modeling choices in `src/`.

Sections:
1. Setup
2. Load & validate the dataset
3. Schema, missing values, duplicates
4. Rating distribution
5. Review-text length
6. The native `categories` column — why we cluster instead
7. Per-product review counts
8. Takeaways

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make `src/` importable when running the notebook from its own folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import PRIMARY_DATASET, RAW_DATA_DIR
from src.preprocessing import load_reviews_csv, validate_columns

pd.set_option("display.max_colwidth", 120)

## 2. Load & validate the dataset

We deliberately use only the canonical Datafiniti CSV. `validate_columns` enforces the four required fields up-front so a wrong file fails fast.

In [ ]:
csv_path = RAW_DATA_DIR / PRIMARY_DATASET
raw = pd.read_csv(csv_path, low_memory=False)
validate_columns(raw.columns)
print(f"Raw shape: {raw.shape}")
raw.head(3)

## 3. Schema, missing values, duplicates

In [ ]:
schema = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
})
schema.loc[["name", "reviews.text", "reviews.rating", "categories"]]

In [ ]:
dup_key = ["name", "reviews.text", "reviews.rating"]
n_dupes = raw.duplicated(subset=dup_key).sum()
print(f"Exact duplicate rows on {dup_key}: {n_dupes}")

Now load through the project's preprocessing pipeline so we can compare raw vs. cleaned counts.

In [ ]:
clean = load_reviews_csv(csv_path)
print(f"Raw rows:    {len(raw):>6}")
print(f"Clean rows:  {len(clean):>6}")
print(f"Dropped:     {len(raw) - len(clean):>6}")
clean.head(3)

## 4. Rating distribution

Rating skew matters for two reasons: it tells us how imbalanced the rating-derived sentiment ground truth will be, and it tells us whether the classifier is going to default to predicting "positive" for everything.

In [ ]:
rating_counts = clean["reviews.rating"].value_counts().sort_index()
print(rating_counts)
fig, ax = plt.subplots(figsize=(6, 3))
rating_counts.plot.bar(ax=ax, color="#3b82f6")
ax.set_title("Rating distribution (cleaned)")
ax.set_xlabel("Stars")
ax.set_ylabel("Reviews")
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment-class balance under the project mapping (1-2 neg / 3 neutral / 4-5 pos).
from src.sentiment import sentiment_from_rating

label_balance = clean["reviews.rating"].map(sentiment_from_rating).value_counts(normalize=True).round(3)
label_balance

## 5. Review-text length

Token length informs `max_length` for DistilBERT. The 95th-percentile word count is a good practical upper bound.

In [ ]:
word_count = clean["reviews.text"].str.split().str.len()
print(word_count.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
fig, ax = plt.subplots(figsize=(6, 3))
word_count.clip(upper=word_count.quantile(0.99)).plot.hist(bins=40, ax=ax, color="#10b981")
ax.set_title("Review length (words, clipped at p99)")
ax.set_xlabel("Words")
plt.tight_layout()
plt.show()

## 6. Native `categories` — why we cluster instead

The brief tells us to cluster into 4–6 meta-categories. The original column has long comma-separated paths and a long tail of unique values, which is why we ignore it for modeling and only keep it for input-shape validation.

In [ ]:
raw_cat_unique = raw["categories"].astype(str).str.strip().nunique()
print(f"Distinct raw category strings: {raw_cat_unique}")
print("\nTop 10 raw category paths:")
raw["categories"].astype(str).value_counts().head(10)

## 7. Per-product review counts

A handful of products dominate review volume. This is why the scoring formula normalizes review count *inside each cluster* — otherwise a single high-volume product would always win.

In [ ]:
per_product = clean.groupby("name").size().sort_values(ascending=False)
print(f"Products: {len(per_product)}  |  median reviews/product: {per_product.median():.0f}")
per_product.head(10)

## 8. Takeaways

- The cleaning pipeline drops a meaningful fraction of rows — duplicates, missing text, out-of-range ratings.
- The class balance is heavily positive-skewed; macro-F1 is the metric to watch in evaluation, not raw accuracy.
- 95th-percentile review length sits well within DistilBERT's 256-token window.
- The native `categories` column is too sparse to use directly — clustering on review-text embeddings is justified.
- Review volume per product is heavy-tailed; in-cluster normalization in the scoring formula prevents a single popular product from dominating every category.